# LoRA Fine-Tuning Whisper Small for Shami (Levantine Arabic)

Fine-tune `openai/whisper-small` using **LoRA** for **Shami / Levantine Arabic** (Syria, Lebanon, Jordan, Palestine).

**Datasets (all open, lightweight download):**

| Dataset | Download Size | Strategy |
|---------|--------------|----------|
| `halabi2016/arabic_speech_corpus` | ~1 GB | Full download (Damascian Levantine) |
| `pain/MASC` | **Streaming** (0 GB disk) | Stream + filter Levantine only |
| `fsicoli/common_voice_22_0` | ~3 GB (Arabic) | Full download |

> MASC is ~200 GB total, but we **stream** it and only keep Levantine samples. No full download needed.

---
## 1. Setup

In [22]:
# Pin datasets<4.0 to avoid script-loader deprecation issues
!pip install -q "transformers>=4.39.0" "datasets>=2.18.0,<4.0.0" "accelerate>=0.26.0"
!pip install -q evaluate jiwer librosa soundfile tensorboard
!pip install -q "peft>=0.9.0" "bitsandbytes>=0.41.0"

In [23]:
import torch, numpy as np, re, os
import datasets as ds_lib
from dataclasses import dataclass
from typing import Any, Dict, List, Union

print(f"PyTorch:  {torch.__version__}")
print(f"Datasets: {ds_lib.__version__}")
print(f"CUDA:     {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:      {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")

PyTorch:  2.10.0+cu128
Datasets: 3.6.0
CUDA:     True
GPU:      NVIDIA A100-SXM4-40GB (42.4 GB)


In [24]:
from huggingface_hub import notebook_login
notebook_login()

---
## 2. Config

In [26]:
MODEL_ID = "openai/whisper-small"
LANGUAGE = "arabic"
LANGUAGE_CODE = "ar"
TASK = "transcribe"

# Datasets toggle
USE_ARABIC_SPEECH_CORPUS = True   # ~1 GB download, Damascian Levantine
USE_MASC_STREAMING = True         # 0 GB disk — streams + filters Levantine only
USE_COMMON_VOICE = True           # ~3 GB download, Arabic subset

# How many Levantine samples to collect from MASC streaming
MASC_MAX_LEVANTINE_SAMPLES = 3000  # Adjust based on your patience/bandwidth

MAX_TRAIN_SAMPLES = 50000  # None = all
MAX_EVAL_SAMPLES = 5000

# LoRA
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "v_proj"]

# Training
OUTPUT_DIR = "./whisper-small-shami-lora"
NUM_TRAIN_EPOCHS = 3
PER_DEVICE_TRAIN_BATCH_SIZE = 8
PER_DEVICE_EVAL_BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 2
LEARNING_RATE = 1e-3
WARMUP_STEPS = 500
FP16 = True
LOGGING_STEPS = 25
EVAL_STEPS = 500
SAVE_STEPS = 500
GENERATION_MAX_LENGTH = 225

HUB_MODEL_ID = None  # e.g. "your-username/whisper-small-shami-lora"

print("Config loaded!")

Config loaded!


---
## 3. Processor & Tokenizer

In [27]:
from transformers import WhisperProcessor, WhisperTokenizer, WhisperFeatureExtractor

feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_ID)
tokenizer = WhisperTokenizer.from_pretrained(MODEL_ID, language=LANGUAGE, task=TASK)
processor = WhisperProcessor.from_pretrained(MODEL_ID, language=LANGUAGE, task=TASK)

print(f"Sampling rate: {feature_extractor.sampling_rate} Hz")

Sampling rate: 16000 Hz


---
## 4. Arabic Text Normalization

In [28]:
def normalize_arabic_text(text: str) -> str:
    if not text: return ""
    text = re.sub(r'[\u0617-\u061A\u064B-\u0652]', '', text)
    text = re.sub(r'[\u0622\u0623\u0625]', '\u0627', text)
    text = re.sub(r'\u0629', '\u0647', text)
    text = re.sub(r'\u0640', '', text)
    text = re.sub(r'[^\w\s\u0600-\u06FF]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print(normalize_arabic_text("بِسْمِ اللَّهِ الرَّحْمَنِ الرَّحِيمِ"))

بسم الله الرحمن الرحيم


---
## 5. Load Datasets

In [29]:
from datasets import load_dataset, Audio, concatenate_datasets, Dataset, DatasetDict

train_parts = []
eval_parts = []

### 5a. Arabic Speech Corpus (~1 GB — Damascian Levantine)

In [30]:
if USE_ARABIC_SPEECH_CORPUS:
    print("Loading halabi2016/arabic_speech_corpus...")
    try:
        asc = load_dataset("halabi2016/arabic_speech_corpus", trust_remote_code=True)
    except RuntimeError as e:
        if "scripts are no longer supported" in str(e):
            print("   Using parquet fallback...")
            asc = DatasetDict({
                "train": load_dataset("halabi2016/arabic_speech_corpus", revision="refs/convert/parquet", split="train"),
                "test": load_dataset("halabi2016/arabic_speech_corpus", revision="refs/convert/parquet", split="test"),
            })
        else: raise e

    for s in asc: print(f"   {s}: {len(asc[s])} samples")

    def prep_asc(b):
        b["sentence"] = normalize_arabic_text(str(b.get("orthographic", b.get("text", ""))))
        return b

    for s in asc:
        asc[s] = asc[s].map(prep_asc)
        asc[s] = asc[s].remove_columns([c for c in asc[s].column_names if c not in ["audio", "sentence"]])

    if "train" in asc: train_parts.append(asc["train"])
    if "test" in asc: eval_parts.append(asc["test"])
    print("   Done!")

Loading halabi2016/arabic_speech_corpus...
   train: 1813 samples
   test: 100 samples
   Done!


### 5b. MASC — Streamed + Levantine Filter (0 GB disk!)

MASC is ~200 GB. Instead of downloading it all, we **stream** it and collect
only Levantine samples on-the-fly until we have enough.

In [31]:
if USE_MASC_STREAMING:
    print(f"Streaming pain/MASC — collecting up to {MASC_MAX_LEVANTINE_SAMPLES} Levantine samples...")
    print("   (No full download — streams from HuggingFace)")

    try:
        masc_stream = load_dataset("pain/MASC", split="train", streaming=True, trust_remote_code=True)
    except RuntimeError as e:
        if "scripts are no longer supported" in str(e):
            masc_stream = load_dataset("pain/MASC", split="train", streaming=True, revision="refs/convert/parquet")
        else: raise e

    # Peek at columns
    first = next(iter(masc_stream))
    print(f"   Columns: {list(first.keys())}")

    # Levantine filter keywords
    lev_kw = ["levantine", "lev", "syrian", "lebanese", "jordanian", "palestinian",
              "syria", "lebanon", "jordan", "palestine", "SY", "LB", "JO", "PS"]

    def is_levantine(example):
        for col in ["dialect", "country", "accent", "type"]:
            val = str(example.get(col, "")).lower()
            if val and any(kw.lower() in val for kw in lev_kw):
                return True
        return False

    # Check if MASC has dialect metadata
    has_dialect = any(k in first for k in ["dialect", "country", "accent"])
    if not has_dialect:
        print("   No dialect column found — will collect general Arabic samples")

    # Stream and collect Levantine samples
    collected = []
    seen = 0
    for example in masc_stream:
        seen += 1
        # Filter for Levantine if metadata exists, otherwise take all
        if has_dialect and not is_levantine(example):
            continue

        text = str(example.get("sentence", example.get("text", example.get("transcription", ""))))
        text = normalize_arabic_text(text)
        if not text.strip():
            continue

        collected.append({
            "audio": example["audio"],
            "sentence": text,
        })

        if len(collected) % 500 == 0:
            print(f"   Collected {len(collected)}/{MASC_MAX_LEVANTINE_SAMPLES} (scanned {seen:,} total)")

        if len(collected) >= MASC_MAX_LEVANTINE_SAMPLES:
            break

    print(f"   Collected {len(collected)} samples (scanned {seen:,} total)")

    if collected:
        # Convert to regular Dataset
        masc_ds = Dataset.from_dict({
            "audio": [c["audio"] for c in collected],
            "sentence": [c["sentence"] for c in collected],
        })

        # 90/10 train/eval split
        masc_split = masc_ds.train_test_split(test_size=0.1, seed=42)
        train_parts.append(masc_split["train"])
        eval_parts.append(masc_split["test"])
        print(f"   Train: {len(masc_split['train'])} | Eval: {len(masc_split['test'])}")

    print("   Done!")
else:
    print("Skipping MASC")

Streaming pain/MASC — collecting up to 3000 Levantine samples...
   (No full download — streams from HuggingFace)
   Columns: ['video_id', 'start', 'end', 'duration', 'text', 'type', 'file_path', 'audio']
   No dialect column found — will collect general Arabic samples
   Collected 500/3000 (scanned 500 total)
   Collected 1000/3000 (scanned 1,000 total)
   Collected 1500/3000 (scanned 1,500 total)
   Collected 2000/3000 (scanned 2,000 total)
   Collected 2500/3000 (scanned 2,500 total)
   Collected 3000/3000 (scanned 3,000 total)
   Collected 3000 samples (scanned 3,000 total)
   Train: 2700 | Eval: 300
   Done!


### 5c. Common Voice 22 (~3 GB Arabic)

In [32]:
if USE_COMMON_VOICE:
    print("Loading fsicoli/common_voice_22_0 (Arabic)...")
    try:
        cv_train = load_dataset("fsicoli/common_voice_22_0", "ar", split="train", trust_remote_code=True)
        cv_test = load_dataset("fsicoli/common_voice_22_0", "ar", split="test", trust_remote_code=True)
    except RuntimeError as e:
        if "scripts are no longer supported" in str(e):
            cv_train = load_dataset("fsicoli/common_voice_22_0", "ar", split="train", revision="refs/convert/parquet")
            cv_test = load_dataset("fsicoli/common_voice_22_0", "ar", split="test", revision="refs/convert/parquet")
        else: raise e

    print(f"   Train: {len(cv_train)} | Test: {len(cv_test)}")

    def prep_cv(b):
        b["sentence"] = normalize_arabic_text(b["sentence"])
        return b

    cv_train = cv_train.map(prep_cv)
    cv_test = cv_test.map(prep_cv)
    cols_rm = [c for c in cv_train.column_names if c not in ["audio", "sentence"]]
    cv_train = cv_train.remove_columns(cols_rm)
    cv_test = cv_test.remove_columns(cols_rm)

    train_parts.append(cv_train)
    eval_parts.append(cv_test)
    print("   Done!")

Loading fsicoli/common_voice_22_0 (Arabic)...
   Train: 28531 | Test: 10500
   Done!


### 5d. Combine & Resample

In [33]:
assert train_parts, "No train data!"
assert eval_parts, "No eval data!"

# Cast ALL parts to the same Audio feature BEFORE concatenating
# This fixes mismatches between streamed (raw dict) and downloaded (Audio feature) datasets
for i in range(len(train_parts)):
    if "audio" in train_parts[i].column_names:
        train_parts[i] = train_parts[i].cast_column("audio", Audio(sampling_rate=16000))

for i in range(len(eval_parts)):
    if "audio" in eval_parts[i].column_names:
        eval_parts[i] = eval_parts[i].cast_column("audio", Audio(sampling_rate=16000))

train_dataset = concatenate_datasets(train_parts)
eval_dataset = concatenate_datasets(eval_parts)

train_dataset = train_dataset.filter(lambda x: len(x["sentence"].strip()) > 0)
eval_dataset = eval_dataset.filter(lambda x: len(x["sentence"].strip()) > 0)

if MAX_TRAIN_SAMPLES:
    train_dataset = train_dataset.shuffle(seed=42).select(range(min(MAX_TRAIN_SAMPLES, len(train_dataset))))
if MAX_EVAL_SAMPLES:
    eval_dataset = eval_dataset.shuffle(seed=42).select(range(min(MAX_EVAL_SAMPLES, len(eval_dataset))))

print(f"Final — Train: {len(train_dataset):,} | Eval: {len(eval_dataset):,}")

Final — Train: 33,044 | Eval: 5,000


In [34]:
s = train_dataset[0]
print(f"Duration: {len(s['audio']['array'])/s['audio']['sampling_rate']:.1f}s")
print(f"Text:     {s['sentence']}")

Duration: 2.4s
Text:     الماء بارد


---
## 6. Feature Extraction

In [35]:
def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_features"] = feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]
    batch["labels"] = tokenizer(batch["sentence"]).input_ids
    return batch

print("Extracting features...")
train_dataset = train_dataset.map(prepare_dataset, remove_columns=train_dataset.column_names, num_proc=1)
eval_dataset = eval_dataset.map(prepare_dataset, remove_columns=eval_dataset.column_names, num_proc=1)
print("Done!")

Extracting features...


Map:   0%|          | 0/33044 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Done!


---
## 7. Data Collator

In [36]:
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    def __call__(self, features):
        inp = [{"input_features": f["input_features"]} for f in features]
        lab = [{"input_ids": f["labels"]} for f in features]
        batch = self.processor.feature_extractor.pad(inp, return_tensors="pt")
        labels_batch = self.processor.tokenizer.pad(lab, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

---
## 8. WER Metric

In [37]:
import evaluate
wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = tokenizer.pad_token_id
    p = [normalize_arabic_text(t) for t in tokenizer.batch_decode(pred_ids, skip_special_tokens=True)]
    r = [normalize_arabic_text(t) for t in tokenizer.batch_decode(label_ids, skip_special_tokens=True)]
    return {"wer": 100 * wer_metric.compute(predictions=p, references=r)}

---
## 9. Model + LoRA

In [38]:
from transformers import WhisperForConditionalGeneration
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)

model.config.use_cache = False  # Required for gradient checkpointing

# Generation params — must be on generation_config, NOT model.config
model.generation_config.forced_decoder_ids = None
model.generation_config.suppress_tokens = []
model.generation_config.language = LANGUAGE
model.generation_config.task = TASK
model.generation_config.forced_decoder_ids = processor.get_decoder_prompt_ids(language=LANGUAGE, task=TASK)

print(f"Base: {MODEL_ID} ({model.num_parameters():,} params)")

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Base: openai/whisper-small (241,734,912 params)


In [39]:
model = get_peft_model(model, LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGET_MODULES,
    lora_dropout=LORA_DROPOUT, bias="none",
))
model.print_trainable_parameters()

trainable params: 3,538,944 || all params: 245,273,856 || trainable%: 1.4429


---
## 10. Train

In [40]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    args=Seq2SeqTrainingArguments(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        learning_rate=LEARNING_RATE,
        warmup_steps=WARMUP_STEPS,
        num_train_epochs=NUM_TRAIN_EPOCHS,
        fp16=FP16,
        eval_strategy="steps", eval_steps=EVAL_STEPS,
        save_strategy="steps", save_steps=SAVE_STEPS,
        logging_steps=LOGGING_STEPS,
        load_best_model_at_end=True,
        metric_for_best_model="wer", greater_is_better=False,
        generation_max_length=GENERATION_MAX_LENGTH,
        predict_with_generate=True,
        remove_unused_columns=False,
        label_names=["labels"],
        report_to=["tensorboard"],
        save_total_limit=3,
        push_to_hub=HUB_MODEL_ID is not None,
        hub_model_id=HUB_MODEL_ID,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
    ),
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
)

print(f"Batch: {PER_DEVICE_TRAIN_BATCH_SIZE}x{GRADIENT_ACCUMULATION_STEPS} = {PER_DEVICE_TRAIN_BATCH_SIZE*GRADIENT_ACCUMULATION_STEPS} | Epochs: {NUM_TRAIN_EPOCHS}")

Batch: 8x2 = 16 | Epochs: 3


In [41]:
train_result = trainer.train()
print("\nDone!")
for k, v in train_result.metrics.items(): print(f"  {k}: {v}")

Step,Training Loss,Validation Loss,Wer
500,1.160342,0.500775,39.813587
1000,0.976839,0.489329,40.503164
1500,0.899788,0.459732,37.638768
2000,0.853361,0.446455,37.043913
2500,0.728488,0.443102,36.899936
3000,0.707266,0.410259,33.751373
3500,0.699017,0.395678,33.357330
4000,0.582821,0.383099,31.440155
4500,0.391267,0.371010,30.674800
5000,0.406626,0.363339,30.587656


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensA


Done!
  train_runtime: 27861.7194
  train_samples_per_second: 3.558
  train_steps_per_second: 0.222
  total_flos: 2.911323244363776e+19
  train_loss: 0.7413590998371096
  epoch: 3.0


In [42]:
model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
sz = sum(os.path.getsize(os.path.join(OUTPUT_DIR, f)) for f in os.listdir(OUTPUT_DIR) if f.endswith(('.bin','.safetensors'))) / 1024**2
print(f"Adapter: {OUTPUT_DIR} (~{sz:.1f} MB)")

Adapter: ./whisper-small-shami-lora (~13.5 MB)


---
## 11. Evaluate

In [43]:
m = trainer.evaluate()
print(f"WER: {m['eval_wer']:.2f}% | Loss: {m['eval_loss']:.4f}")

KeyboardInterrupt: 

---
## 12. Inference

In [44]:
from peft import PeftModel, PeftConfig
from transformers import WhisperForConditionalGeneration, WhisperProcessor, pipeline

# Load + merge LoRA
pc = PeftConfig.from_pretrained(OUTPUT_DIR)
base = WhisperForConditionalGeneration.from_pretrained(pc.base_model_name_or_path)
merged = PeftModel.from_pretrained(base, OUTPUT_DIR).merge_and_unload()

MERGED_DIR = "./whisper-small-shami-merged"
merged.save_pretrained(MERGED_DIR)
WhisperProcessor.from_pretrained(OUTPUT_DIR).save_pretrained(MERGED_DIR)
print(f"Merged model: {MERGED_DIR}")

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged model: ./whisper-small-shami-merged


In [51]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import torch

processor = WhisperProcessor.from_pretrained(MERGED_DIR)
model = WhisperForConditionalGeneration.from_pretrained(MERGED_DIR).to("cuda")

for i in range(min(3, len(td))):
    s = td[i]

    audio = s["audio"]["array"]

    inputs = processor(audio, sampling_rate=16000, return_tensors="pt").to("cuda")

    pred_ids = model.generate(**inputs)
    text = processor.batch_decode(pred_ids, skip_special_tokens=True)[0]

    print(f"\nRef:  {s.get('orthographic', s.get('text', ''))}")
    print(f"Pred: {text}")

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.



Ref:  >atAHat lilbA}iEi lmutajaw~ili >an yakuwna jA*iban lilmuwATini l>aqal~i daxlan
Pred: ataAHato lilobaAiEi Alomu tajawili ano yakuwna jaAibaA lilmuwaAtini AloaqaAlaA na

Ref:  >aHrazat muntaxabAtu lbarAziyli wa>lmAnyA waruwsyA - fawzan fiy muqAbalAtihim l<iEdAdiy~api l~atiy >uqiymat istiEdAdan linihA}iy~Ati ka>si lEAlam - >al~atiy satanTaliqu baEda >aqal~i min >usbuwE
Pred: aHoraZato munotaxabaAtu AlobaraAziyli waalmaAnyaA waruwsoyaA aWozaA fiy mnuAlaAdihim alaEaAdiyapi Alatiy uqiymato asoAdaA daAna linahaAiyaAti kaosi AloEaAlamo alatiy saA AnoTaliqu baEa aqali mino usobuwE

Ref:  >axfaqa majlisu ln~uw~Abi ll~ubnAniy~u fiy xtiyAri ra}iysin jadiydin lilbilAdi - xalafan lilr~a}iysi lHAliy~i l~a*iy tantahiy wilAyatuhu fiy lxAmisi wAlEi$riyn - min mAyuw >ayAra lmuqbil
Pred: axofaqa majolisu AlnuaAbi AlllAniyu fiyAAraAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaAaA

In [52]:
# Your own audio:
# result = pipe("path/to/shami.wav", generate_kwargs={"language": "arabic"}, chunk_length_s=30)
# print(result["text"])

---
## 13. Base vs Fine-Tuned Comparison

In [55]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from jiwer import wer as jiwer_wer
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

base_processor = WhisperProcessor.from_pretrained(MODEL_ID)
base_model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID).to(device)

ft_processor = WhisperProcessor.from_pretrained(MERGED_DIR)
ft_model = WhisperForConditionalGeneration.from_pretrained(MERGED_DIR).to(device)


def infer(model, processor, audio):
    inputs = processor(
        audio,
        sampling_rate=16000,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        pred_ids = model.generate(
            **inputs,
            language="arabic",
            task="transcribe"
        )

    text = processor.batch_decode(pred_ids, skip_special_tokens=True)[0]
    return text


def eval_model(model, processor, ds, n=50):
    preds, refs = [], []

    for i in range(min(n, len(ds))):
        s = ds[i]

        audio = s["audio"]["array"]

        pred = infer(model, processor, audio)

        ref = normalize_arabic_text(
            s.get("orthographic", s.get("text", ""))
        )

        if ref.strip():
            preds.append(normalize_arabic_text(pred))
            refs.append(ref)

    return jiwer_wer(refs, preds) * 100


bw = eval_model(base_model, base_processor, td)
fw = eval_model(ft_model, ft_processor, td)

print(f"Base WER:       {bw:.2f}%")
print(f"Fine-Tuned WER: {fw:.2f}%")
print(f"Improvement:    {bw-fw:.2f} pp")

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Base WER:       100.29%
Fine-Tuned WER: 83.38%
Improvement:    16.91 pp


---
## 14. Push to Hub

In [48]:
HUB_MODEL_ID='mabahboh/whisper-shami'
if HUB_MODEL_ID:
    model.push_to_hub(HUB_MODEL_ID)
    processor.push_to_hub(HUB_MODEL_ID)
    print(f"https://huggingface.co/{HUB_MODEL_ID}")
else:
    print("Set HUB_MODEL_ID to push.")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   4%|3         |  553kB / 14.2MB            

README.md: 0.00B [00:00, ?B/s]

https://huggingface.co/mabahboh/whisper-shami


In [56]:
from transformers import WhisperForConditionalGeneration, WhisperProcessor

HUB_MODEL_ID = "mabahboh/whisper-shami"
MERGED_PATH = "/content/whisper-small-shami-merged"

# Load merged model
model = WhisperForConditionalGeneration.from_pretrained(MERGED_PATH)
processor = WhisperProcessor.from_pretrained(MERGED_PATH)

# Push to HuggingFace Hub
model.push_to_hub(HUB_MODEL_ID)
processor.push_to_hub(HUB_MODEL_ID)

print(f"https://huggingface.co/{HUB_MODEL_ID}")

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...akbmiwu/model.safetensors:   5%|4         | 46.6MB /  967MB            

https://huggingface.co/mabahboh/whisper-shami


---
## Tips

- **More data:** `MAX_TRAIN_SAMPLES = None` + `MASC_MAX_LEVANTINE_SAMPLES = 10000`
- **Better results:** `LORA_R=64`, add `k_proj, out_proj`, 5-10 epochs
- **Less VRAM:** Enable 8-bit (Section 9), batch 4 + grad accum 4
- **Bigger model:** Change to `whisper-medium` or `whisper-large-v3`